
# Step 0.0 — Environment management with uv

**Purpose.** Create a reproducible Python environment so that every figure and
table in this project can be regenerated by someone else from a clean clone.

`uv` is used instead of conda: it resolves and installs from a lockfile in
seconds, and it pins transitive dependencies, which conda's `environment.yml`
does not do by default.

This notebook is documentation with runnable checks. The shell commands are shown
for copying into a terminal; only the verification cells are meant to be executed
inside the kernel.


## 1. Install uv

```bash
# Linux / macOS / WSL
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows PowerShell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"

# or, if you already have pip
pip install uv
```

## 2. Create the environment and install dependencies

From the repository root:

```bash
uv venv --python 3.11
source .venv/bin/activate            # Windows: .venv\Scripts\Activate.ps1

uv pip install -r requirements.txt   # exact pinned versions
# or, for a fresh resolve from pyproject.toml:
uv pip install -e .
```

## 3. Register the Jupyter kernel

```bash
python -m ipykernel install --user --name retina --display-name "Python 3 (retina)"
```

Then pick **Python 3 (retina)** as the kernel in JupyterLab. If the notebooks run
against the system Python instead, every version number recorded below is wrong.

## 4. Re-pin after any change

```bash
uv pip freeze > requirements.txt
git add requirements.txt && git commit -m "Update pinned dependencies"
```

## 5. Run non-interactively

```bash
uv run jupyter lab
uv run python tools/build_notebooks.py
```


## Package list and why each one is here

| Package | Why it is needed |
|---|---|
| `scanpy` | the analysis framework: QC, normalisation, PCA/UMAP, Leiden, DE |
| `anndata` | the on-disk and in-memory data structure (`.h5ad`) |
| `numpy`, `pandas`, `scipy` | arrays, tables, sparse matrices |
| `matplotlib`, `seaborn` | figures |
| `jupyter`, `ipykernel` | notebook interface and kernel registration |
| `gseapy` | GO / pathway enrichment on marker lists (the Python stand-in for the paper's topGO) |
| `igraph`, `leidenalg` | **required backend for `sc.tl.leiden`** — Scanpy does not vendor a Leiden implementation, so clustering fails without them |
| `h5py` | `.h5ad` read/write (pulled in by anndata, pinned so checkpoints stay loadable) |

Nothing else is installed by default. `harmonypy` is needed **only** if the Step 04
batch diagnostics justify integration; installing it is a deliberate act that must
be recorded in the Methods section, not a default.

In [1]:
import sys
from pathlib import Path

# Make the repository root importable regardless of where Jupyter was launched.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts").exists(), (
    f"Cannot locate the repository root from {Path.cwd()}. "
    "Launch JupyterLab from the repository root."
)
sys.path.insert(0, str(REPO_ROOT))

from scripts import config as cfg
from scripts import io_utils, qc, preprocessing, clustering, annotation, egfp_analysis, plotting

cfg.ensure_directories()
plotting.set_style()
SEED = io_utils.set_seeds()
print(f"Repository root: {REPO_ROOT}")
print(f"Random seed: {SEED}")

Repository root: D:\retina\project1_zebrafish_retina
Random seed: 0


In [2]:
# Verify the environment and write the reproducibility record.
versions_path = io_utils.write_versions()
print(f"\nSession info written to {versions_path.relative_to(cfg.REPO_ROOT)}")

  python       3.11.15
  platform     Windows-10-10.0.26200-SP0
  scanpy       1.11.5
  anndata      0.11.4
  numpy        2.4.6
  pandas       2.3.3
  scipy        1.17.1
  matplotlib   3.11.1
  random_seed  0
  leidenalg    0.12.0

Session info written to docs\session_info.json


D:\retina\project1_zebrafish_retina\scripts\io_utils.py:325: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  "scanpy": sc.__version__,


In [3]:
# Confirm the Leiden backend is importable BEFORE spending an hour on preprocessing.
try:
    import leidenalg  # noqa: F401
    import igraph     # noqa: F401
    print("Leiden backend available.")
except ImportError as exc:
    raise ImportError(
        "sc.tl.leiden needs 'leidenalg' and 'igraph'. Install with:\n"
        "    uv pip install leidenalg igraph\n"
        "and re-pin with: uv pip freeze > requirements.txt"
    ) from exc

Leiden backend available.



### What to check before continuing

- [ ] The active kernel is **Python 3 (retina)**, not the system Python.
- [ ] `docs/session_info.json` exists and lists a Scanpy version.
- [ ] The Leiden backend imported without error.
- [ ] `data/` contains the eight sample folders (checked in Step 01).